In [1]:
import sys
sys.path.append("../pipelines")
from aq import *
import pandas as pd

print("Lot :", BATCH)

Lot : L-2026-10-01


In [2]:
SOURCES = [("crm", "client", "crm_client"), ("crm", "contrat", "crm_contrat"), ("crm", "type_contrat", "crm_type_contrat"),
           ("crm", "sinistre", "crm_sinistre"), ("cif", "clients", "cif_clients"), ("cif", "compte", "cif_compte"),
           ("cif", "pays", "cif_pays")]

def ingerer(base, table, cible):
    src = reader(base)
    with src.connect() as c:
        df = pd.read_sql(text(f"SELECT * FROM {table}"), c, coerce_float=False).astype("string")
    df = df.astype(object).where(df.notna(), None)
    h = pd.util.hash_pandas_object(df.fillna("\\N").astype(str), index=False)
    df["_row_hash"] = [f"{v:016x}" for v in h]
    df["_batch_id"] = BATCH
    df["_source_system"] = base
    with eng().begin() as c:
        c.execute(text(f"DELETE FROM bronze.{cible} WHERE _batch_id = :b"), {"b": BATCH})
        df.to_sql(cible, c, schema="bronze", if_exists="append", index=False, chunksize=10_000)
    return len(df)

def ingerer_tout():
    executer("INSERT INTO dq.lot_ingestion (batch_id, statut) VALUES (:b, 'EN_COURS') "
             "ON DUPLICATE KEY UPDATE statut='EN_COURS', finished_at=NULL", b=BATCH)
    for base, table, cible in SOURCES:
        print(f"{base}.{table:<13} -> bronze.{cible:<17} {ingerer(base, table, cible):>7} lignes")
    executer("UPDATE dq.lot_ingestion SET statut='TERMINE', finished_at=NOW() WHERE batch_id = :b", b=BATCH)

ingerer_tout()

crm.client        -> bronze.crm_client         100000 lignes
crm.contrat       -> bronze.crm_contrat        100000 lignes
crm.type_contrat  -> bronze.crm_type_contrat       12 lignes
crm.sinistre      -> bronze.crm_sinistre       100000 lignes
cif.clients       -> bronze.cif_clients         40000 lignes
cif.compte        -> bronze.cif_compte          45000 lignes
cif.pays          -> bronze.cif_pays              250 lignes


In [3]:
def etat_bronze():
    res = []
    for base, table, cible in SOURCES:
        n_src = int(lire(f"SELECT COUNT(*) AS n FROM {base}.{table}").n[0])
        r = lire(f"""SELECT COUNT(*) AS n,
                            SUM(_batch_id = :b) AS n_lot,
                            COUNT(DISTINCT _batch_id) AS lots,
                            SUM(_batch_id IS NULL OR _batch_id = '' OR _source_system IS NULL OR _source_system = ''
                                OR _ingested_at IS NULL OR _row_hash IS NULL OR _row_hash = '') AS meta_ko
                     FROM bronze.{cible}""", b=BATCH)
        res.append({"table_bronze": cible, "lignes_source": n_src, "lignes_bronze": int(r.n[0]),
                    "lignes_du_lot": int(r.n_lot[0] or 0), "lots": int(r.lots[0]),
                    "ecart": int(r.n[0]) - n_src, "metadonnees_manquantes": int(r.meta_ko[0] or 0)})
    return pd.DataFrame(res)

avant = etat_bronze()
print(avant.to_string(index=False))
print(lire("SELECT * FROM dq.lot_ingestion WHERE batch_id = :b", b=BATCH).to_string(index=False))

    table_bronze  lignes_source  lignes_bronze  lignes_du_lot  lots  ecart  metadonnees_manquantes
      crm_client         100000         100000         100000     1      0                       0
     crm_contrat         100000         100000         100000     1      0                       0
crm_type_contrat             12             12             12     1      0                       0
    crm_sinistre         100000         100000         100000     1      0                       0
     cif_clients          40000          40000          40000     1      0                       0
      cif_compte          45000          45000          45000     1      0                       0
        cif_pays            250            250            250     1      0                       0
    batch_id          started_at         finished_at  statut
L-2026-10-01 2026-10-14 14:09:47 2026-10-20 20:51:23 TERMINE


In [4]:
print(lire("SELECT bronze_row_id, client_id, cp, _batch_id, _source_system, _ingested_at, _row_hash FROM bronze.crm_client LIMIT 5").to_string(index=False))
print(lire("""SELECT column_name, data_type FROM information_schema.columns
              WHERE table_schema = 'bronze' AND table_name = 'crm_contrat' ORDER BY ordinal_position""").to_string(index=False))

 bronze_row_id client_id    cp    _batch_id _source_system        _ingested_at        _row_hash
        100001    A-1000 69300 L-2026-10-01            crm 2026-10-20 20:51:02 8fe9e59d3f5268ea
        100002   A-10000 73000 L-2026-10-01            crm 2026-10-20 20:51:02 bf96f99ffb10ed08
        100003   A-10001 71000 L-2026-10-01            crm 2026-10-20 20:51:02 5471aff6d3a407b8
        100004   A-10002 69009 L-2026-10-01            crm 2026-10-20 20:51:02 4ac5023b52ef397a
        100005   A-10003 69100 L-2026-10-01            crm 2026-10-20 20:51:02 e33662a2cdbedbb2
             column_name data_type
           bronze_row_id    bigint
              contrat_id   varchar
         souscripteur_id   varchar
               assure_id   varchar
               code_type   varchar
              date_debut   varchar
                date_fin   varchar
     duree_garantie_mois   varchar
          prime_annuelle   varchar
date_derniere_cotisation   varchar
               _batch_id   varchar
    

In [5]:
ingerer_tout()
apres = etat_bronze()
comparaison = avant[["table_bronze", "lignes_bronze"]].merge(apres[["table_bronze", "lignes_bronze"]], on="table_bronze",
                                                             suffixes=("_1er_passage", "_2e_passage"))
comparaison["doublons_crees"] = comparaison.lignes_bronze_2e_passage - comparaison.lignes_bronze_1er_passage
print(comparaison.to_string(index=False))
print("Doublons créés par la relance :", int(comparaison.doublons_crees.sum()))

crm.client        -> bronze.crm_client         100000 lignes
crm.contrat       -> bronze.crm_contrat        100000 lignes
crm.type_contrat  -> bronze.crm_type_contrat       12 lignes
crm.sinistre      -> bronze.crm_sinistre       100000 lignes
cif.clients       -> bronze.cif_clients         40000 lignes
cif.compte        -> bronze.cif_compte          45000 lignes
cif.pays          -> bronze.cif_pays              250 lignes
    table_bronze  lignes_bronze_1er_passage  lignes_bronze_2e_passage  doublons_crees
      crm_client                     100000                    100000               0
     crm_contrat                     100000                    100000               0
crm_type_contrat                         12                        12               0
    crm_sinistre                     100000                    100000               0
     cif_clients                      40000                     40000               0
      cif_compte                      45000              

In [6]:
with reader("crm").connect() as c:
    flottant = pd.read_sql(text("SELECT contrat_id, prime_annuelle FROM contrat"), c)
    exact = pd.read_sql(text("SELECT contrat_id, prime_annuelle FROM contrat"), c, coerce_float=False)
compar = pd.DataFrame({"contrat_id": exact.contrat_id, "lecture_par_defaut": flottant.prime_annuelle.astype(str),
                       "lecture_coerce_float_False": exact.prime_annuelle.astype(str)})
diff = compar[compar.lecture_par_defaut != compar.lecture_coerce_float_False]
print("Valeurs différentes selon la lecture :", len(diff), "sur", len(compar))
print(diff.head(5).to_string(index=False))

Valeurs différentes selon la lecture : 9912 sur 100000
contrat_id lecture_par_defaut lecture_coerce_float_False
 CT0000020              184.1                     184.10
 CT0000036             1352.1                    1352.10
 CT0000062              326.6                     326.60
 CT0000069              119.6                     119.60
 CT0000115              342.6                     342.60


In [7]:
def somme_controle(base, table, ou=""):
    colonnes = lire("SELECT column_name FROM information_schema.columns WHERE table_schema = :s AND table_name = :t AND column_name NOT LIKE '\\_%' "
                    "AND column_name <> 'bronze_row_id' ORDER BY ordinal_position", s=base, t=table).column_name.tolist()
    liste = ", ".join(colonnes)
    r = lire(f"SELECT COUNT(*) AS n, COALESCE(SUM(CRC32(CONCAT_WS('|', {liste}))), 0) AS crc FROM {base}.{table} {ou}")
    return int(r.n[0]), int(r.crc[0])

ordre = [("crm", "client", "crm_client"), ("crm", "contrat", "crm_contrat"), ("crm", "sinistre", "crm_sinistre"),
         ("crm", "type_contrat", "crm_type_contrat"), ("cif", "clients", "cif_clients"), ("cif", "compte", "cif_compte"),
         ("cif", "pays", "cif_pays")]

porte_b = []
for base, table, cible in ordre:
    n_src, crc_src = somme_controle(base, table)
    n_brz, crc_brz = somme_controle("bronze", cible, f"WHERE _batch_id = '{BATCH}'")
    porte_b.append({"table": cible, "n_src": n_src, "n_brz": n_brz, "ecart_n": n_brz - n_src, "ecart_crc": crc_brz - crc_src})
    mesurer("BRONZE", "RQ-021", cible, n_src, abs(n_src - n_brz) + int(crc_src != crc_brz))

res = pd.DataFrame(porte_b)
fmt = lambda n: f"{int(n):,}".replace(",", " ")
lignes_md = ["| Table bronze | Lignes source | Lignes bronze | Écart de comptage | Écart de somme de contrôle |", "|---|---|---|---|---|"]
for r in res.itertuples(index=False):
    lignes_md.append(f"| {r.table} | {fmt(r.n_src)} | {fmt(r.n_brz)} | {r.ecart_n} | {r.ecart_crc} |")
print("\n".join(lignes_md))

| Table bronze | Lignes source | Lignes bronze | Écart de comptage | Écart de somme de contrôle |
|---|---|---|---|---|
| crm_client | 100 000 | 100 000 | 0 | 0 |
| crm_contrat | 100 000 | 100 000 | 0 | 0 |
| crm_sinistre | 100 000 | 100 000 | 0 | 0 |
| crm_type_contrat | 12 | 12 | 0 | 0 |
| cif_clients | 40 000 | 40 000 | 0 | 0 |
| cif_compte | 45 000 | 45 000 | 0 | 0 |
| cif_pays | 250 | 250 | 0 | 0 |


In [8]:
def epreuve(alteration):
    executer("DROP TABLE IF EXISTS bronze.crm_client_test")
    executer("CREATE TABLE bronze.crm_client_test LIKE bronze.crm_client")
    executer("INSERT INTO bronze.crm_client_test SELECT * FROM bronze.crm_client WHERE _batch_id = :b", b=BATCH)
    if alteration:
        executer(alteration)
    n_t, crc_t = somme_controle("bronze", "crm_client_test")
    n_s, crc_s = somme_controle("crm", "client")
    executer("DROP TABLE bronze.crm_client_test")
    return {"ecart_comptage": n_t - n_s, "ecart_somme_controle": crc_t - crc_s,
            "porte_B": "OK" if n_t == n_s and crc_t == crc_s else "ÉCHEC"}

essais = {
    "copie non modifiée": None,
    "une valeur modifiée (code postal)": "UPDATE bronze.crm_client_test SET cp = 'XXXXX' ORDER BY bronze_row_id LIMIT 1",
    "une ligne supprimée": "DELETE FROM bronze.crm_client_test ORDER BY bronze_row_id LIMIT 1",
    "une ligne en double": "INSERT INTO bronze.crm_client_test (client_id, type_personne, nom, prenom, raison_sociale, date_naissance, adresse, cp, ville, pays, telephone, email, date_creation, _batch_id, _source_system, _row_hash) "
                           "SELECT client_id, type_personne, nom, prenom, raison_sociale, date_naissance, adresse, cp, ville, pays, telephone, email, date_creation, _batch_id, _source_system, _row_hash "
                           "FROM bronze.crm_client_test ORDER BY bronze_row_id LIMIT 1",
}
resultats = pd.DataFrame([{"alteration": k, **epreuve(v)} for k, v in essais.items()])
print(resultats.to_string(index=False))

                       alteration  ecart_comptage  ecart_somme_controle porte_B
               copie non modifiée               0                     0      OK
une valeur modifiée (code postal)               0            1745534541   ÉCHEC
              une ligne supprimée              -1           -1539251045   ÉCHEC
              une ligne en double               1            1539251045   ÉCHEC
